In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
# Load the CSV from your Drive folder
df = pd.read_csv('/content/drive/MyDrive/sql/Dataset.csv')

# Quick look
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (3900, 18)

Columns: ['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases']


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [3]:
print(df.info())
print("\nNull Values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [4]:
# Strip whitespace from column names (dataset has trailing spaces)
df.columns = df.columns.str.strip()

# Confirm fix
print(df.columns.tolist())

['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Review Rating', 'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Payment Method', 'Frequency of Purchases']


In [5]:
# Strip whitespace from all string values
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

# Encode Yes/No columns as 1/0
df['Discount Applied'] = df['Discount Applied'].map({'Yes': 1, 'No': 0})
df['Promo Code Used']  = df['Promo Code Used'].map({'Yes': 1, 'No': 0})
df['Subscription Status'] = df['Subscription Status'].map({'Yes': 1, 'No': 0})

print("Binary encoding done.")
df[['Discount Applied','Promo Code Used','Subscription Status']].head()

Binary encoding done.


,Discount Applied,Promo Code Used,Subscription Status
0,1,1,1
1,1,1,1
2,1,1,1
3,1,1,1
4,1,1,1


In [6]:
# Map purchase frequency to a numeric score (higher = more frequent)
frequency_map = {
    'Weekly': 7,
    'Bi-Weekly': 6,
    'Fortnightly': 5,
    'Monthly': 4,
    'Every 3 Months': 3,
    'Quarterly': 3,
    'Annually': 1
}

df['Frequency Score'] = df['Frequency of Purchases'].map(frequency_map)

print(df['Frequency Score'].value_counts())

Frequency Score
3    1147
1     572
4     553
6     547
5     542
7     539
Name: count, dtype: int64


In [7]:
# Loyalty = how often they buy + how many times before + are they subscribed
# All 3 signals normalized to same scale

df['Loyalty Score'] = (
    (df['Previous Purchases'] / df['Previous Purchases'].max()) * 50 +  # 50% weight
    (df['Frequency Score'] / df['Frequency Score'].max()) * 30 +        # 30% weight
    (df['Subscription Status']) * 20                                     # 20% weight
).round(2)

print(df['Loyalty Score'].describe())

count    3900.000000
mean       48.323628
std        19.063091
min         5.290000
25%        33.965000
50%        48.140000
75%        61.860000
max       100.000000
Name: Loyalty Score, dtype: float64


In [8]:
# If a customer ALWAYS uses promo + discount → high dependency
# Both columns are identical in this dataset, so we use a combined flag

df['Promo Dependency Score'] = (
    df['Discount Applied'] + df['Promo Code Used']  # max = 2, min = 0
)

# 0 = no promo used, 1 = partial, 2 = full promo dependency
print(df['Promo Dependency Score'].value_counts())

Promo Dependency Score
0    2223
2    1677
Name: count, dtype: int64


In [9]:
# Value = spend per transaction × purchase history (lifetime proxy)
df['Lifetime Value Proxy'] = (
    df['Purchase Amount (USD)'] * df['Previous Purchases']
).round(2)

# Segment into tiers using quartiles
df['Value Tier'] = pd.qcut(
    df['Lifetime Value Proxy'],
    q=4,
    labels=['Low', 'Mid', 'High', 'Premium']
)

print(df['Value Tier'].value_counts())

Value Tier
Mid        977
Low        975
Premium    975
High       973
Name: count, dtype: int64


In [10]:
# Review Rating above 4.0 = satisfied customer
df['Satisfaction Flag'] = (df['Review Rating'] >= 4.0).astype(int)

print("Satisfied customers:", df['Satisfaction Flag'].sum())
print("Unsatisfied customers:", (df['Satisfaction Flag'] == 0).sum())

Satisfied customers: 1634
Unsatisfied customers: 2266


In [11]:
# Save cleaned + enriched dataset back to your Drive
output_path = '/content/drive/MyDrive/sql/Dataset_Cleaned.csv'
df.to_csv(output_path, index=False)

print(f"✅ Saved! Shape: {df.shape}")
print(f"New columns added: Frequency Score, Loyalty Score, Promo Dependency Score, Lifetime Value Proxy, Value Tier, Satisfaction Flag")

✅ Saved! Shape: (3900, 24)
New columns added: Frequency Score, Loyalty Score, Promo Dependency Score, Lifetime Value Proxy, Value Tier, Satisfaction Flag


In [12]:
# SQLite comes built-in with Python — no installation needed!
import sqlite3
import pandas as pd

# Load your cleaned dataset
df = pd.read_csv('/content/drive/MyDrive/sql/Dataset_Cleaned.csv')

# Create a local SQLite database
conn = sqlite3.connect('customer_analysis.db')

# Push dataframe into SQL table
df.to_sql('customers', conn, if_exists='replace', index=False)

print("✅ Table created! Rows:", pd.read_sql("SELECT COUNT(*) as total FROM customers", conn).iloc[0,0])

✅ Table created! Rows: 3900


In [13]:
# This lets you run any SQL query and see results as a table
def run_sql(query):
    return pd.read_sql(query, conn)

In [15]:
run_sql("""
    SELECT
        "Value Tier",
        COUNT(*) AS customer_count,
        ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
        ROUND(AVG("Previous Purchases"), 2) AS avg_prev_purchases,
        ROUND(AVG("Loyalty Score"), 2) AS avg_loyalty,
        ROUND(AVG("Promo Dependency Score"), 2) AS avg_promo_dependency,
        ROUND(AVG("Review Rating"), 2) AS avg_rating,
        SUM("Subscription Status") AS subscribers
    FROM customers
    GROUP BY "Value Tier"
    ORDER BY avg_spend DESC
""")

,Value Tier,customer_count,avg_spend,avg_prev_purchases,avg_loyalty,avg_promo_dependency,avg_rating,subscribers
0,Premium,975,79.66,39.72,62.65,0.84,3.79,263
1,High,973,60.37,30.68,54.25,0.89,3.73,278
2,Mid,977,50.46,22.75,45.72,0.91,3.73,277
3,Low,975,48.59,8.28,30.69,0.80,3.75,235


In [16]:
run_sql("""
    SELECT
        CASE
            WHEN "Promo Dependency Score" = 0 AND "Loyalty Score" > 60 THEN 'Genuinely Loyal'
            WHEN "Promo Dependency Score" = 2 AND "Loyalty Score" < 40 THEN 'Promo Hunter'
            WHEN "Promo Dependency Score" = 2 AND "Loyalty Score" >= 40 THEN 'Loyal but Promo Reliant'
            ELSE 'Occasional Buyer'
        END AS customer_segment,
        COUNT(*) AS customer_count,
        ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
        ROUND(AVG("Loyalty Score"), 2) AS avg_loyalty,
        ROUND(AVG("Review Rating"), 2) AS avg_rating
    FROM customers
    GROUP BY customer_segment
    ORDER BY customer_count DESC
""")

,customer_segment,customer_count,avg_spend,avg_loyalty,avg_rating
0,Occasional Buyer,1869,60.05,37.90,3.74
1,Loyal but Promo Reliant,1297,59.33,63.69,3.72
2,Promo Hunter,380,59.12,29.36,3.82
3,Genuinely Loyal,354,60.57,67.40,3.83


In [17]:
run_sql("""
    SELECT
        Season,
        Category,
        CASE
            WHEN "Previous Purchases" <= 10 THEN 'New Customer'
            WHEN "Previous Purchases" BETWEEN 11 AND 30 THEN 'Mid Tenure'
            ELSE 'High Tenure'
        END AS tenure_group,
        COUNT(*) AS customer_count,
        ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend
    FROM customers
    GROUP BY Season, Category, tenure_group
    ORDER BY Season, customer_count DESC
""")

,Season,Category,tenure_group,customer_count,avg_spend
0,Fall,Clothing,Mid Tenure,180,63.35
1,Fall,Clothing,High Tenure,164,59.48
2,Fall,Accessories,Mid Tenure,130,58.95
3,Fall,Accessories,High Tenure,124,63.17
4,Fall,Clothing,New Customer,83,60.99
5,Fall,Accessories,New Customer,70,62.53
6,Fall,Footwear,Mid Tenure,62,63.94
7,Fall,Footwear,High Tenure,46,60.43
8,Fall,Outerwear,High Tenure,34,59.62
9,Fall,Outerwear,Mid Tenure,32,59.16


In [18]:
run_sql("""
    SELECT
        Location,
        COUNT(*) AS total_customers,
        ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
        ROUND(AVG("Promo Dependency Score"), 2) AS avg_promo_dependency,
        ROUND(AVG("Loyalty Score"), 2) AS avg_loyalty,
        CASE
            WHEN AVG("Promo Dependency Score") < 0.8
             AND AVG("Purchase Amount (USD)") > 60 THEN '🟢 Organic Hotspot'
            WHEN AVG("Promo Dependency Score") >= 1.5 THEN '🔴 Discount Driven'
            ELSE '🟡 Mixed'
        END AS market_type
    FROM customers
    GROUP BY Location
    HAVING total_customers >= 30
    ORDER BY avg_spend DESC
    LIMIT 20
""")

,Location,total_customers,avg_spend,avg_promo_dependency,avg_loyalty,market_type
0,Alaska,72,67.60,0.81,51.25,🟡 Mixed
1,Pennsylvania,74,66.57,0.89,51.57,🟡 Mixed
2,Arizona,65,66.55,0.68,50.96,🟢 Organic Hotspot
3,West Virginia,81,63.88,0.99,48.30,🟡 Mixed
4,Nevada,87,63.38,0.94,50.52,🟡 Mixed
5,Washington,73,63.33,0.88,47.24,🟡 Mixed
6,North Dakota,83,62.89,0.92,46.28,🟡 Mixed
7,Virginia,77,62.88,0.75,47.55,🟢 Organic Hotspot
8,Utah,71,62.58,0.93,49.28,🟡 Mixed
9,Michigan,73,62.10,0.79,47.36,🟢 Organic Hotspot


In [19]:
run_sql("""
    SELECT
        CASE
            WHEN Age BETWEEN 18 AND 30 THEN '18-30'
            WHEN Age BETWEEN 31 AND 45 THEN '31-45'
            WHEN Age BETWEEN 46 AND 60 THEN '46-60'
            ELSE '60+'
        END AS age_group,
        Gender,
        "Payment Method",
        "Subscription Status",
        COUNT(*) AS customer_count,
        ROUND(AVG("Loyalty Score"), 2) AS avg_loyalty,
        ROUND(AVG("Purchase Amount (USD)"), 2) AS avg_spend,
        ROUND(AVG("Promo Dependency Score"), 2) AS avg_promo_dependency,
        ROUND(AVG("Satisfaction Flag"), 2) AS satisfaction_rate
    FROM customers
    WHERE "Value Tier" = 'Premium'
    GROUP BY age_group, Gender, "Payment Method", "Subscription Status"
    ORDER BY avg_loyalty DESC
    LIMIT 15
""")

,age_group,Gender,Payment Method,Subscription Status,customer_count,avg_loyalty,avg_spend,avg_promo_dependency,satisfaction_rate
0,60+,Male,PayPal,1,6,82.93,72.33,2.0,0.50
1,18-30,Male,PayPal,1,16,81.75,77.38,2.0,0.50
2,31-45,Male,Credit Card,1,13,81.44,78.77,2.0,0.46
3,18-30,Male,Venmo,1,10,81.26,77.00,2.0,0.30
4,46-60,Male,Venmo,1,11,80.98,80.09,2.0,0.18
5,18-30,Male,Cash,1,7,79.98,80.57,2.0,0.71
6,60+,Male,Cash,1,12,79.22,82.08,2.0,0.33
7,18-30,Male,Debit Card,1,12,78.67,88.25,2.0,0.67
8,46-60,Male,Debit Card,1,16,78.64,82.44,2.0,0.50
9,31-45,Male,PayPal,1,15,77.74,77.40,2.0,0.40


In [20]:
# Save all 5 query results as CSVs for Power BI
queries = {
    'q1_value_tiers': """SELECT "Value Tier", COUNT(*) AS customer_count, ROUND(AVG("Purchase Amount (USD)"),2) AS avg_spend, ROUND(AVG("Loyalty Score"),2) AS avg_loyalty, ROUND(AVG("Promo Dependency Score"),2) AS avg_promo_dependency FROM customers GROUP BY "Value Tier" """,

    'q2_segments': """SELECT CASE WHEN "Promo Dependency Score"=0 AND "Loyalty Score">60 THEN 'Genuinely Loyal' WHEN "Promo Dependency Score"=2 AND "Loyalty Score"<40 THEN 'Promo Hunter' WHEN "Promo Dependency Score"=2 AND "Loyalty Score">=40 THEN 'Loyal but Promo Reliant' ELSE 'Occasional Buyer' END AS customer_segment, COUNT(*) AS customer_count, ROUND(AVG("Purchase Amount (USD)"),2) AS avg_spend FROM customers GROUP BY customer_segment""",

    'q3_season_category': """SELECT Season, Category, CASE WHEN "Previous Purchases"<=10 THEN 'New' WHEN "Previous Purchases" BETWEEN 11 AND 30 THEN 'Mid' ELSE 'High' END AS tenure_group, COUNT(*) AS customer_count, ROUND(AVG("Purchase Amount (USD)"),2) AS avg_spend FROM customers GROUP BY Season, Category, tenure_group""",

    'q4_geography': """SELECT Location, COUNT(*) AS total_customers, ROUND(AVG("Purchase Amount (USD)"),2) AS avg_spend, ROUND(AVG("Promo Dependency Score"),2) AS avg_promo_dependency, ROUND(AVG("Loyalty Score"),2) AS avg_loyalty FROM customers GROUP BY Location HAVING total_customers >= 30""",

    'q5_ideal_customer': """SELECT CASE WHEN Age BETWEEN 18 AND 30 THEN '18-30' WHEN Age BETWEEN 31 AND 45 THEN '31-45' WHEN Age BETWEEN 46 AND 60 THEN '46-60' ELSE '60+' END AS age_group, Gender, "Payment Method", COUNT(*) AS customer_count, ROUND(AVG("Loyalty Score"),2) AS avg_loyalty, ROUND(AVG("Purchase Amount (USD)"),2) AS avg_spend FROM customers WHERE "Value Tier"='Premium' GROUP BY age_group, Gender, "Payment Method" ORDER BY avg_loyalty DESC"""
}

for name, query in queries.items():
    result = pd.read_sql(query, conn)
    path = f'/content/drive/MyDrive/sql/{name}.csv'
    result.to_csv(path, index=False)
    print(f"✅ Saved: {name}.csv — {len(result)} rows")

✅ Saved: q1_value_tiers.csv — 4 rows
✅ Saved: q2_segments.csv — 4 rows
✅ Saved: q3_season_category.csv — 48 rows
✅ Saved: q4_geography.csv — 50 rows
✅ Saved: q5_ideal_customer.csv — 48 rows
